# PixiJoy Flow (M4) — Colab GPU (quality)

Runtime → **GPU** → **Run all**.

- **Google Drive is ON** — checkpoints + quality data live under `MyDrive/pixijoy_ml/flow/` so disconnects do **not** force a full restart.
- Auto-fetches real training data (no tensorflow_datasets / no HF).
- Reconnect → Run all again: skips finished data prep, **auto-resumes** from `*.ckpt.keras` + `train_state.json`.
- Pin TFLite only if verify prints **OK**.
- Set `USE_DRIVE = False` only if you refuse Drive (ephemeral `/content`).


In [ ]:
!nvidia-smi || true
import tensorflow as tf
print("TF", tf.__version__, "GPUs", tf.config.list_physical_devices("GPU"))
assert tf.config.list_physical_devices('GPU'), 'Select Runtime → GPU before Run all'


In [ ]:
USE_DRIVE = True  # persist ckpt + data on Drive (survive disconnect)

import os, urllib.request, zipfile, shutil
from pathlib import Path
from google.colab import files

ROOT = Path('/content/pixijoy_flow')
DRIVE = Path('/content/drive/MyDrive/pixijoy_ml/flow')
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    (DRIVE / 'dist').mkdir(parents=True, exist_ok=True)
    (DRIVE / 'data').mkdir(parents=True, exist_ok=True)

def _link_to_drive(name: str) -> None:
    """Point local name/ at Drive so writes survive runtime death."""
    if not USE_DRIVE:
        Path(name).mkdir(parents=True, exist_ok=True)
        return
    local = Path(name)
    target = DRIVE / name
    target.mkdir(parents=True, exist_ok=True)
    if local.exists() or local.is_symlink():
        if local.is_symlink():
            if local.resolve() == target.resolve():
                print(f'{name} -> Drive OK')
                return
            local.unlink()
        elif local.is_dir():
            for p in local.rglob('*'):
                if not p.is_file():
                    continue
                dest = target / p.relative_to(local)
                dest.parent.mkdir(parents=True, exist_ok=True)
                if not dest.is_file():
                    shutil.copy2(p, dest)
            shutil.rmtree(local)
        else:
            local.unlink()
    local.symlink_to(target)
    print(f'{name} -> {target}')

ZIP_URL = 'https://raw.githubusercontent.com/Saikumar-S0906/Release-/main/flow_lab.zip'
ok = False
try:
    urllib.request.urlretrieve(ZIP_URL, 'flow_lab.zip')
    with zipfile.ZipFile('flow_lab.zip') as z:
        z.extractall('.')
    ok = Path('train.py').is_file() and Path('prepare_quality_data.py').is_file()
    print('lab zip OK', ok)
except Exception as e:
    print('zip fetch failed:', e)
if not ok:
    print('Upload flow_lab.zip')
    up = files.upload()
    name = next(iter(up))
    with zipfile.ZipFile(name) as z:
        z.extractall('.')
    ok = Path('train.py').is_file()
assert ok, 'lab zip missing train.py / prepare_quality_data.py — refresh Runtime and re-fetch'
_link_to_drive('dist')
_link_to_drive('data')


In [ ]:
from pathlib import Path
print('prepare_quality_data (real data; skips if data/READY already on Drive)…')
rc = get_ipython().system('python -u prepare_quality_data.py --data ./data')
assert rc == 0, 'prepare_quality_data failed'
assert (Path('data') / 'READY').is_file(), 'data/READY missing — Drive link or prepare failed'
print('data READY ok')


In [ ]:
from pathlib import Path
import hashlib
USE_DRIVE = globals().get('USE_DRIVE', True)
OUT = Path('/content/drive/MyDrive/pixijoy_ml/flow/dist/flow.keras') if USE_DRIVE else Path('dist') / 'flow.keras'
OUT.parent.mkdir(parents=True, exist_ok=True)
ckpt = OUT.with_suffix('.ckpt.keras')
print('ckpt exists' if ckpt.is_file() else 'fresh train', OUT)
print('state', OUT.parent / 'train_state.json', (OUT.parent / 'train_state.json').is_file())

cmd = "python -u train.py --quality --out \"" + str(OUT) + "\""
if ckpt.is_file():
    cmd += " --resume"
    print('will resume from checkpoint')
print('RUN', cmd)
rc = get_ipython().system(cmd)
assert rc == 0, 'train failed'
rc = get_ipython().system('python -u export_tflite.py --keras "' + str(OUT) + '" --out "dist/flow.tflite"')
assert rc == 0, 'export failed (refusing untrained is correct)'
rc = get_ipython().system('python -u verify_contract.py --quality --data ./data')
assert rc == 0, 'verify --quality failed — do NOT pin'
p = Path('dist/flow.tflite')
assert p.is_file(), p
print('SHIP_OK', p.stat().st_size, hashlib.sha256(p.read_bytes()).hexdigest())


In [ ]:
from google.colab import files
files.download('dist/flow.tflite')
print('Attach flow.tflite to ml-packs-v1 ONLY if verify printed OK / SHIP_OK')
